# Phase 1 — Step 1.4: Inventory Shape

**Goal:** understand exactly what is being sold, so Phase 7's solver is designed
against the real problem size and Phase 6 prices a unit that actually exists.

| Question (from the plan) | Answered in |
| --- | --- |
| Screens by city / type / mount position / size | §2 |
| Which types are static vs vehicle-mounted | §2.1 |
| **How many distinct sellable units exist** | §3 |
| What the slot dimension contains, and how a booking references it | §1 |

**Deliverable:** [`docs/results/1.4_inventory_shape.md`](../docs/results/1.4_inventory_shape.md),
regenerable with `python scripts/build_inventory_report.py --strict`.

As in notebook 01, all logic lives in `src/agentiq/data/inventory.py` — this notebook
orchestrates and renders. The primitives built here (`occupancy_timeline`,
`CapacityModel`, the as-of date) are the same ones Steps 1.5 and 6.1 need.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def _project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "agentiq").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project tree.")


ROOT = _project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

%load_ext autoreload
%autoreload 2

from agentiq.data import DataLake, ProjectPaths
from agentiq.data.inventory import (
    availability_by_block,
    block_demand,
    cold_start_census,
    concentration,
    deployment_split,
    facet_counts,
    infer_as_of_date,
    measure_capacity,
    occupancy_timeline,
    profile_inventory,
    rotation_economics,
    sellable_units,
    solver_scenarios,
)
from agentiq.data.inventory_report import render_inventory_report

PATHS = ProjectPaths(ROOT).ensure_dirs()
lake = DataLake(PATHS.raw_data, cache_dir=PATHS.cache)

screens = lake["screens"]
bookings = lake["bookings"]
slot_dim = lake["dim_slot"]
print(f"{len(screens):,} screens · {len(bookings):,} booking lines · {len(slot_dim)} time blocks")

11,163 screens · 191,109 booking lines · 6 time blocks


## 1 · What one row of inventory is

### 1.1 The slot dimension, and how a booking references it

In [2]:
display(slot_dim)

# A booking references the slot dimension by time_block_id and *also* carries a
# denormalised `daypart`. Is that column redundant, or does it disagree anywhere?
joined = bookings[["time_block_id", "daypart"]].merge(
    slot_dim[["time_block_id", "nearest_daypart"]], on="time_block_id", how="left"
)
mismatches = int(
    (joined["daypart"].astype("object") != joined["nearest_daypart"].astype("object")).sum()
)
print(f"bookings.daypart vs dim_slot.nearest_daypart: {mismatches} mismatches in {len(joined):,} lines")
print("-> daypart is a pure denormalisation; drop it and key on time_block_id.")
print("\nNote night maps to TWO blocks, so daypart is not a key:")
display(slot_dim.groupby("nearest_daypart", observed=True)["time_block_id"].apply(list))

,time_block_id,time_block_label,start_hour,end_hour,nearest_daypart
0,1,00:00-04:00,0,4,night
1,2,04:00-08:00,4,8,morning
2,3,08:00-12:00,8,12,midday
3,4,12:00-16:00,12,16,afternoon
4,5,16:00-20:00,16,20,evening
5,6,20:00-24:00,20,24,night


bookings.daypart vs dim_slot.nearest_daypart: 0 mismatches in 191,109 lines
-> daypart is a pure denormalisation; drop it and key on time_block_id.

Note night maps to TWO blocks, so daypart is not a key:


nearest_daypart
afternoon       [4]
evening         [5]
midday          [3]
morning         [2]
night        [1, 6]
Name: time_block_id, dtype: object

### 1.2 The as-of date — the dataset's "today"

There is no `today` column. Without one, a back-test would train on its own future,
and "available inventory" would be meaningless. The date is triangulated from the
booking-status windows and cross-checked against the ridership series.

In [3]:
display(
    bookings.groupby("booking_status", observed=True).agg(
        lines=("booking_id", "count"),
        min_start=("start_date", "min"),
        max_end=("end_date", "max"),
    )
)

as_of = infer_as_of_date(bookings, lake["ridership_actuals"])
print(f"as-of date: {as_of.date.date()}   unambiguous: {as_of.is_unambiguous}\n")
for evidence in as_of.evidence():
    print(f"  - {evidence}")

,lines,min_start,max_end
booking_status,,,
active,29954,2026-02-21,2027-02-14
completed,111727,2025-08-19,2026-08-18
upcoming,49428,2026-08-20,2027-02-21


as-of date: 2026-08-19   unambiguous: True

  - last `completed` booking ends 2026-08-18
  - first `upcoming` booking starts 2026-08-20, leaving exactly one day between them
  - all 29,954 of 29,954 `active` lines span it
  - ridership actuals stop at 2026-08-19


### 1.3 Capacity: how many slots a screen × block × date holds

This is the crux of the whole optimisation — it turns a screen from a yes/no purchase
into a divisible resource. It is **measured**, not configured.

In [4]:
# rotation_type and slots_booked_per_day are not independent facts.
display(pd.crosstab(bookings["rotation_type"], bookings["slots_booked_per_day"], margins=True))

capacity = measure_capacity(bookings, slot_dim)
display(capacity.rotation_table())
print(
    f"blocks/day={capacity.blocks_per_day}  slots/block={capacity.slots_per_block}  "
    f"slot-units per screen per day={capacity.slots_per_screen_day}"
)

slots_booked_per_day,1,2,3,4,5,6,All
rotation_type,,,,,,,
full_exclusivity,0,0,0,0,15711,17138,32849
partial_rotation,0,43000,27132,23400,0,0,93532
single_rotation,64728,0,0,0,0,0,64728
All,64728,43000,27132,23400,15711,17138,191109


,rotation_type,slots_booked_per_day,min_slots,max_slots
0,single_rotation,1,1,1
1,partial_rotation,"2, 3, 4",2,4
2,full_exclusivity,"5, 6",5,6


blocks/day=6  slots/block=6  slot-units per screen per day=36


In [5]:
# Proof that 6 is a real ceiling and not an artefact of the sample.
#
# The naive check expands every booking to one row per unit per date (~12M rows).
# The sweep line emits +slots at start and -slots at end+1 instead: two events per
# booking regardless of flight length. Same answer, and it scales.
timeline = occupancy_timeline(bookings)
print(f"{len(bookings):,} bookings -> {len(timeline):,} events (vs ~12M rows expanded)")

peak = timeline.groupby(["screen_id", "time_block_id"], observed=True)["occupied_slots"].max()
display(peak.value_counts().sort_index().rename("units at this peak").to_frame())
print(f"max concurrent slots anywhere: {int(peak.max())}")
print(f"units ever oversold:           {int((peak > capacity.slots_per_block).sum())}")

assert capacity.is_validated, "capacity model not validated by the sweep line"
sold_out = int((peak == capacity.slots_per_block).sum())
print(
    f"\n{sold_out:,} of {peak.size:,} booked units ({sold_out / peak.size:.1%}) have hit "
    f"full capacity at some point -> scarcity is real, not theoretical."
)

191,109 bookings -> 375,407 events (vs ~12M rows expanded)


,units at this peak
occupied_slots,
1,1661
2,2824
3,3638
4,4349
5,4295
6,25137


max concurrent slots anywhere: 6
units ever oversold:           0

25,137 of 41,904 booked units (60.0%) have hit full capacity at some point -> scarcity is real, not theoretical.


## 2 · What the inventory is

In [6]:
deployment = deployment_split(screens)
display(deployment)

# Does screen_type alone determine static vs mobile? If so, downstream code never
# needs to null-check location_id / vehicle_id.
per_type = deployment.groupby("screen_type", observed=True)["mounting"].nunique()
assert (per_type == 1).all(), f"screen_type spans both mountings: {per_type[per_type > 1]}"
print("screen_type -> mounting is 1:1; screen_type is sufficient to pick the D1 model.")

,screen_type,mounting,screens,share_of_network,exposure_model
0,metro_station,static,6391,0.572516,D1 static (zone + POI + stop throughput)
1,bus_stop,static,2157,0.193228,D1 static (zone + POI + stop throughput)
2,metro_rail_coach,mobile,1400,0.125414,D1 mobile (journey)
3,bus,mobile,1215,0.108842,D1 mobile (journey)


screen_type -> mounting is 1:1; screen_type is sufficient to pick the D1 model.


In [7]:
for facet, by in (("screen_type", "city_id"), ("position", "screen_type"), ("screen_size", "screen_type")):
    print(f"\n=== {facet} x {by}")
    display(facet_counts(screens, facet, by))


=== screen_type x city_id


city_id,ACS,DAT,LH,all
screen_type,,,,
bus,396,414,405,1215
bus_stop,702,720,735,2157
metro_rail_coach,136,324,940,1400
metro_station,502,1665,4224,6391
all,1736,3123,6304,11163



=== position x screen_type


screen_type,bus,bus_stop,metro_rail_coach,metro_station,all
position,,,,,
(null),0,0,1400,0,1400
back,405,0,0,0,405
entrance_exit,0,0,0,1275,1275
left,405,719,0,0,1124
platform,0,0,0,5116,5116
right,405,719,0,0,1124
top,0,719,0,0,719
all,1215,2157,1400,6391,11163



=== screen_size x screen_type


screen_type,bus,bus_stop,metro_rail_coach,metro_station,all
screen_size,,,,,
L,810,171,0,2447,3428
M,0,331,1400,2797,4528
S,405,1655,0,1147,3207
all,1215,2157,1400,6391,11163


In [8]:
# The brief-named inventory: "bus-rear" (brief 1 excludes it, brief 2 requires it)
# and "metro platform boards" (brief 1 requires them). Both must be addressable.
targets = {
    "bus-rear": (screens["screen_type"] == "bus") & (screens["position"] == "back"),
    "metro platform": (screens["screen_type"] == "metro_station") & (screens["position"] == "platform"),
    "metro entrance": (screens["screen_type"] == "metro_station") & (screens["position"] == "entrance_exit"),
    "interior coach": screens["screen_type"] == "metro_rail_coach",
}
display(
    pd.DataFrame(
        [{"brief language": name, "screens": int(mask.sum())} for name, mask in targets.items()]
    )
)
print("All four are directly expressible as filters -> enforceable in the Phase 5 filter.")

,brief language,screens
0,bus-rear,405
1,metro platform,5116
2,metro entrance,1275
3,interior coach,1400


All four are directly expressible as filters -> enforceable in the Phase 5 filter.


In [9]:
# Concentration: screens sharing a location share an audience. This sizes the
# Phase 3 overlap problem.
display(concentration(screens, lake["locations"], lake["vehicles"]))

,grouping,groups,screens,min,median,max
0,per location (all),910,8548,3,3.0,50
1,per location (bus_stop),719,2157,3,3.0,3
2,per location (metro_station),191,6391,10,34.0,50
3,per zone (static screens),30,8548,103,238.0,549
4,per vehicle,854,2615,2,3.0,4
5,per corridor (mobile screens),94,2615,12,18.0,100


## 3 · The sellable-unit count — the optimisation problem size

**This is the number Phase 7's solver design must cite.**

In [10]:
units = sellable_units(
    screens,
    bookings,
    capacity,
    horizon_start=as_of.date,
    horizon_end=bookings["end_date"].max(),
)
print(f"horizon: {units.horizon_start.date()} -> {units.horizon_end.date()} ({units.horizon_days} days)\n")
display(units.per_city)

network = units.network
print(
    f"\nPER DAY, NETWORK: {network['slot_units_per_day']:,} sellable slot-units"
    f"  ({network['screens']:,} screens x {capacity.blocks_per_day} blocks x {capacity.slots_per_block} slots)"
)
print(f"OVER THE HORIZON: {network['slot_units_horizon']:,} slot-units")
print(f"  committed:      {network['committed_slot_units']:,}")
print(f"  still sellable: {network['available_slot_units']:,}")

assert (units.per_city["available_slot_units"] >= 0).all(), "committed exceeds capacity somewhere"

horizon: 2026-08-19 -> 2027-02-21 (187 days)



,city_id,screens,block_units_per_day,slot_units_per_day,slot_units_horizon,committed_slot_units,available_slot_units,utilisation
0,ACS,1736,10416,62496,11686752,828135,10858617,0.070861
1,DAT,3123,18738,112428,21024036,3368569,17655467,0.160225
2,LH,6304,37824,226944,42438528,6256141,36182387,0.147417
3,network,11163,66978,401868,75149316,10452845,64696471,0.139094



PER DAY, NETWORK: 401,868 sellable slot-units  (11,163 screens x 6 blocks x 6 slots)
OVER THE HORIZON: 75,149,316 slot-units
  committed:      10,452,845
  still sellable: 64,696,471


In [11]:
# Where the availability constraint actually bites. A single network average would
# make it look non-binding.
availability = availability_by_block(
    screens,
    bookings,
    capacity,
    slot_dim,
    horizon_start=units.horizon_start,
    horizon_end=units.horizon_end,
)
display(availability)
spread = availability["utilisation"].max() / availability["utilisation"].min()
print(
    f"network average utilisation {network['committed_slot_units'] / network['slot_units_horizon']:.1%}, "
    f"but peak block is {spread:.1f}x the quietest block."
)

,time_block_id,time_block_label,nearest_daypart,capacity_slot_units,committed_slot_units,available_slot_units,utilisation
0,1,00:00-04:00,night,12524886,613943,11910943,0.049018
1,2,04:00-08:00,morning,12524886,3035030,9489856,0.242320
2,3,08:00-12:00,midday,12524886,2007110,10517776,0.160250
3,4,12:00-16:00,afternoon,12524886,1619406,10905480,0.129295
4,5,16:00-20:00,evening,12524886,2542127,9982759,0.202966
5,6,20:00-24:00,night,12524886,635229,11889657,0.050717


network average utilisation 13.9%, but peak block is 4.9x the quietest block.


In [12]:
# What the solver is actually handed, once a campaign scopes the problem.
city_counts = screens.groupby("city_id", observed=True).size()
scenarios = solver_scenarios(
    units,
    largest_city_screens=int(city_counts.max()),
    median_flight_days=int(bookings["duration_days"].median()),
)
display(scenarios)
print(
    "Filtering before scoring is what makes this tractable: the same problem is\n"
    f"{scenarios['decision_pairs'].iloc[0]:,} pairs unscoped and "
    f"{scenarios['decision_pairs'].iloc[-2]:,} after eligibility."
)

,scenario,eligible_screens,blocks,flight_days,decision_pairs,slot_day_inventory
0,"Whole network, whole horizon (upper bound, not...",11163,6,187,66978,75149316
1,"Largest city, all blocks, median flight",6304,6,63,37824,14297472
2,"Largest city, 2 requested blocks, median flight",6304,2,63,12608,4765824
3,After eligibility filter (assumed 10% of a city),630,2,63,1260,476280
4,Hyper-local brief (assumed ~50 screens in radius),50,2,15,100,9000


Filtering before scoring is what makes this tractable: the same problem is
66,978 pairs unscoped and 1,260 after eligibility.


## 4 · What the market actually buys

In [13]:
display(block_demand(bookings, slot_dim))

rotation = rotation_economics(bookings)
display(rotation)
print(
    "Median price per slot falls monotonically as slots rise -> first evidence for the\n"
    "non-linearity nuance. Step 1.5 must test whether it survives controlling for\n"
    "screen type and time block."
)

,time_block_id,time_block_label,nearest_daypart,lines,line_share,slot_days,median_price
0,1,00:00-04:00,night,8544,0.044707,26581,36.09
1,2,04:00-08:00,morning,59475,0.311210,144962,81.81
2,3,08:00-12:00,midday,41340,0.216316,111490,68.95
3,4,12:00-16:00,afternoon,26082,0.136477,72506,64.26
4,5,16:00-20:00,evening,47142,0.246676,124865,85.67
5,6,20:00-24:00,night,8526,0.044613,26703,36.40


,slots_booked_per_day,lines,median_price_per_slot,value,vs_single_slot,implied_total_vs_linear
0,1,64728,76.600,4.852030e+08,0.000000,1.000000
1,2,43000,73.670,5.033732e+08,-0.038251,0.961749
2,3,27132,72.455,3.349479e+08,-0.054112,0.945888
3,4,23400,72.290,3.821294e+08,-0.056266,0.943734
4,5,15711,71.880,3.168887e+08,-0.061619,0.938381
5,6,17138,70.960,4.247625e+08,-0.073629,0.926371


Median price per slot falls monotonically as slots rise -> first evidence for the
non-linearity nuance. Step 1.5 must test whether it survives controlling for
screen type and time block.


In [14]:
cold = cold_start_census(screens, bookings)
display(cold)
print(
    "Cold start is concentrated in one city, not scattered -> the fallback ladder is\n"
    "how a whole market gets priced, and it is the new-city scaling story."
)

,facet,value,screens,with_history,no_history,cold_share
0,city_id,ACS,1736,800,936,0.539171
1,city_id,DAT,3123,2835,288,0.092219
2,city_id,LH,6304,6304,0,0.000000
3,screen_type,bus,1215,893,322,0.265021
4,screen_type,bus_stop,2157,1348,809,0.375058
5,screen_type,metro_rail_coach,1400,1400,0,0.000000
6,screen_type,metro_station,6391,6298,93,0.014552
7,screen_size,L,3428,3126,302,0.088098
8,screen_size,M,4528,4369,159,0.035115
9,screen_size,S,3207,2444,763,0.237917


Cold start is concentrated in one city, not scattered -> the fallback ladder is
how a whole market gets priced, and it is the new-city scaling story.


## 5 · Generate the deliverable

In [15]:
from datetime import datetime, timezone

shape = profile_inventory(lake)
document = render_inventory_report(
    shape, generated_at=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
)

target = PATHS.docs / "results" / "1.4_inventory_shape.md"
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(document, encoding="utf-8")
print(f"wrote {target.relative_to(ROOT)}  ({len(document):,} chars)")

wrote docs\results\1.4_inventory_shape.md  (18,034 chars)


## Step 1.4 exit checklist

The plan's exit criterion: *"a one-page inventory-shape summary including the exact
sellable-unit count per city per day. Solver design in Phase 7 must cite this number."*

In [16]:
gates = {
    "slot dimension is closed and gapless": (
        slot_dim["start_hour"].min() == 0
        and slot_dim["end_hour"].max() == 24
        and (slot_dim.sort_values("time_block_id")["start_hour"].tolist()[1:]
             == slot_dim.sort_values("time_block_id")["end_hour"].tolist()[:-1])
    ),
    "booking->slot reference verified (daypart redundant)": mismatches == 0,
    "as-of date unambiguous": as_of.is_unambiguous,
    "capacity measured and proved by sweep line": capacity.is_validated,
    "screen_type determines static vs mobile": bool((per_type == 1).all()),
    "every screen counted in the facet tables": int(
        facet_counts(screens, "position", "screen_type").loc["all", "all"]
    ) == len(screens),
    "sellable-unit count per city per day computed": len(units.per_city) == screens["city_id"].nunique() + 1,
    "availability never exceeds capacity": bool((units.per_city["available_slot_units"] >= 0).all()),
    "brief-named inventory is addressable": all(int(m.sum()) > 0 for m in targets.values()),
    "1.4 report written": target.is_file(),
}

width = max(len(name) for name in gates)
for name, passed in gates.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name:<{width}}")
print(f"\n{sum(bool(v) for v in gates.values())}/{len(gates)} gates passed.")
print(
    f"\nHEADLINE FOR PHASE 7: {network['slot_units_per_day']:,} sellable slot-units per day "
    f"network-wide;\n{network['slot_units_horizon']:,} over the {units.horizon_days}-day horizon, "
    f"of which {network['available_slot_units']:,} are still available."
)

PASS  slot dimension is closed and gapless                
PASS  booking->slot reference verified (daypart redundant)
PASS  as-of date unambiguous                              
PASS  capacity measured and proved by sweep line          
PASS  screen_type determines static vs mobile             
PASS  every screen counted in the facet tables            
PASS  sellable-unit count per city per day computed       
PASS  availability never exceeds capacity                 
PASS  brief-named inventory is addressable                
PASS  1.4 report written                                  

10/10 gates passed.

HEADLINE FOR PHASE 7: 401,868 sellable slot-units per day network-wide;
75,149,316 over the 187-day horizon, of which 64,696,471 are still available.


## What Step 1.5 inherits

| Artifact | Used by |
| --- | --- |
| `infer_as_of_date()` → `2026-08-19` | every train/test split; settled vs committed classification |
| `occupancy_timeline()` sweep line | Step 1.5 occupancy at date grain, Step 6.1 scarcity |
| `CapacityModel` (6 blocks × 6 slots) | Phase 6 unit pricing, Phase 7 constraints |
| `sellable_units()` / `solver_scenarios()` | Phase 7 solver-strategy choice and latency table |
| `concentration()` | Phase 3 overlap graph |
| `cold_start_census()` | Step 6.5 fallback ladder |
| §4.2 slot-discount finding | Step 1.5's formal non-linearity test |

**Next — Step 1.5 (demand history).** Price distribution by every facet named here,
occupancy at date grain, bundle pricing, and the lost-leads price-gap curve.